# Access Events Audit Logger

**Purpose:** Track governance audit events for compliance evidence and security monitoring.

**Target Table:** `workspace.audit.audit_access_events`

**Event Type:** Governance audit events (request-level telemetry)

**Key Metrics:**
* User/service principal access patterns
* Operation types (READ, WRITE, DELETE, GRANT)
* Access results (SUCCESS, DENIED)
* Object-level tracking (tables, schemas, volumes)

**Usage:**
```python
dbutils.notebook.run(
    "LMIP/notebooks/audit/audit_access_events",
    timeout_seconds=300,
    arguments={
        "actor": "user@company.com",
        "action_type": "READ",
        "object_name": "workspace.silver.customers",
        "object_type": "TABLE",
        "result": "SUCCESS"
    }
)
```

In [0]:
# Notebook parameters - configured via dbutils.widgets or job parameters
dbutils.widgets.text("actor", "", "Actor (user or service principal)")
dbutils.widgets.text("action_type", "READ", "Action Type (READ/WRITE/DELETE/GRANT)")
dbutils.widgets.text("object_name", "", "Object Name (fully qualified)")
dbutils.widgets.text("object_type", "TABLE", "Object Type (TABLE/SCHEMA/VOLUME/etc.)")
dbutils.widgets.text("result", "SUCCESS", "Result (SUCCESS/DENIED)")
dbutils.widgets.text("request_id", "", "Optional: Request ID for correlation")

# Retrieve parameter values
actor = dbutils.widgets.get("actor")
action_type = dbutils.widgets.get("action_type")
object_name = dbutils.widgets.get("object_name")
object_type = dbutils.widgets.get("object_type")
result = dbutils.widgets.get("result")
request_id = dbutils.widgets.get("request_id")

print(f"Access Audit: {actor} | {action_type} | {object_name} | Result: {result}")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import uuid

In [0]:
# Validation: Ensure mandatory parameters are provided
if not actor:
    raise ValueError("Parameter 'actor' is required")
    
if not object_name:
    raise ValueError("Parameter 'object_name' is required")
    
if action_type not in ["READ", "WRITE", "DELETE", "GRANT", "REVOKE", "SELECT", "INSERT", "UPDATE"]:
    raise ValueError(f"Invalid action_type: {action_type}")
    
if object_type not in ["TABLE", "SCHEMA", "VOLUME", "CATALOG", "VIEW", "FUNCTION"]:
    raise ValueError(f"Invalid object_type: {object_type}")
    
if result not in ["SUCCESS", "DENIED"]:
    raise ValueError(f"Invalid result: {result}. Must be SUCCESS or DENIED")

print("✓ Parameter validation passed")

In [0]:
# Generate surrogate key
access_event_sk = int(str(uuid.uuid4().int)[:18])  # 18-digit bigint

# Capture event timestamp
event_time = datetime.now()

# Build access event record
access_event_record = {
    "access_event_sk": access_event_sk,
    "actor": actor,
    "action_type": action_type,
    "object_name": object_name,
    "object_type": object_type,
    "event_time": event_time,
    "result": result
}

print(f"Generated access event record with SK: {access_event_sk}")
print(f"  - Actor: {actor}")
print(f"  - Action: {action_type} on {object_type}")
print(f"  - Object: {object_name}")
print(f"  - Result: {result}")

# Security alert for denied access
if result == "DENIED":
    print(f"⚠️ SECURITY ALERT: Access denied for {actor} attempting {action_type} on {object_name}")

In [0]:
# Create DataFrame from access event record
access_event_df = spark.createDataFrame([access_event_record])

# Write to audit table (append mode)
target_table_name = "workspace.audit.audit_access_events"

try:
    access_event_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table_name)
    
    print(f"✓ Successfully logged access event to {target_table_name}")
    
except Exception as e:
    print(f"✗ Failed to write access event record: {str(e)}")
    raise

In [0]:
%sql
-- Verify the access event was logged successfully
-- Shows recent access audit trail

SELECT 
    actor,
    action_type,
    object_name,
    object_type,
    event_time,
    result,
    CASE 
        WHEN result = 'DENIED' THEN '⚠️ Security Alert'
        ELSE '✓ Authorized'
    END as access_status
FROM workspace.audit.audit_access_events
ORDER BY event_time DESC
LIMIT 10

In [0]:
%sql
-- Security monitoring: Track denied access attempts
-- Critical for identifying unauthorized access attempts

SELECT 
    actor,
    action_type,
    object_name,
    object_type,
    event_time,
    COUNT(*) OVER (PARTITION BY actor, DATE(event_time)) as attempts_today
FROM workspace.audit.audit_access_events
WHERE result = 'DENIED'
  AND event_time >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 20

In [0]:
%sql
-- Analyze access patterns by actor for anomaly detection
-- Useful for identifying unusual access behavior

SELECT 
    actor,
    COUNT(*) as total_accesses,
    COUNT(DISTINCT object_name) as unique_objects_accessed,
    SUM(CASE WHEN result = 'SUCCESS' THEN 1 ELSE 0 END) as successful_accesses,
    SUM(CASE WHEN result = 'DENIED' THEN 1 ELSE 0 END) as denied_accesses,
    MAX(event_time) as last_access_time
FROM workspace.audit.audit_access_events
WHERE event_time >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY actor
ORDER BY total_accesses DESC
LIMIT 20

In [0]:
# Return success indicator with access event details
result_data = {
    "status": "success",
    "access_event_sk": access_event_sk,
    "access_granted": result == "SUCCESS",
    "security_alert": result == "DENIED"
}

dbutils.notebook.exit(result_data)